# Fase C1 · CLONALG corregido — Mac mini

La hipermutación de CLONALG estaba **al revés y a escala de todo el dominio**
(`sigma = exp(-rank) * (U-L)`: la mejor solución mutaba con σ ≈ 0,99·200). Cada clon era casi
un punto al azar y CLONALG perdía contra la búsqueda aleatoria en 53–60 de 71 funciones.
Corregido en `competidores.py` (commit `bb9765b`): α = exp(−ρ·f*), con f* la afinidad normalizada
(1 = la mejor), σ = α·0,1·(U−L), ρ = 5, y el reemplazo toca las d peores.

Hay que repetir **todo lo de CLONALG**: las 6 celdas del protocolo, su curva de convergencia (B1) y
el tiempo (B3, se repite entero para que todos los algoritmos queden medidos en la misma sesión).

| Paso | Qué | Tiempo aprox. |
|---|---|---|
| 1 | chequeo | segundos |
| 2 | respaldar lo viejo | segundos |
| 3 | tiempo B3 — **con la máquina libre** | ~25 min |
| 4 | protocolo 5k y 50k | ~2 h |
| 5 | protocolo 500k | ~15 h (de noche) |
| 6 | convergencia B1 | ~45 min |
| 7 | verificar | segundos |

Copiar al mini: **`competidores.py`** (commit `bb9765b`). Los demás no cambian; la celda 0 lo comprueba.
No correrlo al mismo tiempo que `faseC_alpha` (los dos usan 20 workers).
**Solo en el Mac mini.**

In [ ]:
import sys, os, json, math, shutil, hashlib, subprocess, opfunu
import numpy as np

WORKERS = 20
ESPERADO = {
    "competidores.py":        "f7572ec3102d3038c8a90baf4ed761e1398860ce633dbcef7dc4cce7b1b83428",
    "complementarias.py":     "2889b74aae020d143a4980e19a70ccc4c6e959dceb379f91e178c681cee8af1a",
    "experimento_config.py":  "ef9c5af25d46afa681276efeb5d0e25c68290404dc6093e8d8783b3554a914d1",
    "correr_marea.py":        "64421093bf66e9234f8157d8adfcc118aecb37617e9d5db9f735cbed270b4e6b",
}

def correr(*args):
    """Ejecuta un script con el mismo Python del kernel y muestra la salida en vivo."""
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        if "pkg_resources" not in linea:
            print(linea, end="")
    return p.wait()

print("python :", sys.executable)
print("carpeta:", os.getcwd())
print("opfunu :", opfunu.__version__)
assert opfunu.__version__ == "1.0.4", "opfunu tiene que ser 1.0.4 (1.0.1 esta roto)"
for f, h in ESPERADO.items():
    real = hashlib.sha256(open(f, "rb").read()).hexdigest()
    print(f"{f:24s} {real[:16]}  {'OK' if real == h else 'DISTINTO'}")
    assert real == h, f"{f} no es la version del commit bb9765b: copiala de nuevo"
print("Entorno y archivos OK")

## 1 · Chequeo (segundos)

In [ ]:
rc = correr("correr_marea.py", "--check")
assert rc == 0, "el chequeo de entorno fallo: no seguir, mandar la salida a Claude"
rc = correr("complementarias.py", "check")
assert rc == 0, "el chequeo fallo: no seguir, mandar la salida a Claude"
CHEQUEADO = True

## 2 · Respaldar los resultados viejos de CLONALG
Si no se mueven, el script **reanuda**, los encuentra completos y no recalcula nada.
Van a `resultados_competidores_buggy/clonalg_mutacion_invertida/`. Si esta celda se corre otra vez
(después de una interrupción), no toca lo que ya es nuevo.

In [ ]:
R, C = "resultados_experimento", "resultados_experimento/complementarias"
BK = "resultados_competidores_buggy/clonalg_mutacion_invertida"
os.makedirs(BK, exist_ok=True)
mover = [f"{R}/CLONALG_D{d}_FES{fes}.json" for d in (10, 50) for fes in (5000, 50000, 500000)]
mover += [f"{C}/CONV_CLONALG_D{d}.json" for d in (10, 50)] + [f"{C}/TIEMPO.json"]
for src in mover:
    dst = os.path.join(BK, os.path.basename(src))
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.move(src, dst); print("respaldado", src)
    elif os.path.exists(src):
        print("ya respaldado antes; se deja (es nuevo):", src)
    else:
        print("no esta (ok si ya se respaldo):", src)
RESPALDADO = True

## 3 · Tiempo de cómputo, B3 (~25 min) — **con la máquina libre**
Repite la medición entera (los 10 algoritmos). No correr otras celdas ni otros notebooks mientras tanto.

In [ ]:
assert globals().get("CHEQUEADO") and globals().get("RESPALDADO"), "primero las celdas 1 y 2"
rc = correr("complementarias.py", "tiempo")
assert rc == 0

## 4 · Protocolo de CLONALG a 5k y 50k (~2 h)

In [ ]:
assert globals().get("RESPALDADO"), "primero la celda de respaldo"
for fes in ("5000", "50000"):
    rc = correr("correr_marea.py", "--algo", "CLONALG", "--maxfes", fes, "--workers", str(WORKERS))
    assert rc == 0

## 5 · Protocolo de CLONALG a 500k (~15 h; se puede dejar de noche)

In [ ]:
assert globals().get("RESPALDADO"), "primero la celda de respaldo"
rc = correr("correr_marea.py", "--algo", "CLONALG", "--maxfes", "500000", "--workers", str(WORKERS))
assert rc == 0

## 6 · Convergencia de CLONALG, B1 (~45 min)
Solo recalcula CLONALG (los demás están completos). Compara el final de cada curva con el JSON a 500k
del paso 5, así que va después.

In [ ]:
rc = correr("complementarias.py", "conv", "--workers", str(WORKERS), "--runs", "51")
assert rc == 0

## 7 · Verificar y traer al MacBook
Tiene que terminar en **TODO OK**. La tabla de abajo cuenta en cuántas funciones (de 71) la búsqueda
aleatoria le gana a CLONALG por mediana; antes eran 53–60.

Traer al MacBook (**a esas mismas carpetas, no a la raíz**):
- los 6 `resultados_experimento/CLONALG_D*_FES*.json`,
- `resultados_experimento/complementarias/CONV_CLONALG_D10.json` y `CONV_CLONALG_D50.json`,
- `resultados_experimento/complementarias/TIEMPO.json`.

In [ ]:
rc = correr("complementarias.py", "verificar", "--runs", "51")
print("\nverificacion:", "OK" if rc == 0 else "REVISAR")
R = "resultados_experimento"
for d in (10, 50):
    for fes in (5000, 50000, 500000):
        c = json.load(open(f"{R}/CLONALG_D{d}_FES{fes}.json"))
        r = json.load(open(f"{R}/RANDOM_D{d}_FES{fes}.json"))
        pids = sorted({k.split("_")[0] for k in r})
        med = lambda v, p: np.median([v[f"{p}_run{i}"] for i in range(51)])
        gana = sum(med(r, p) < med(c, p) for p in pids)
        print(f"D{d} {fes:>6}: random gana a CLONALG en {gana}/71 funciones")